In [1]:
import sys
from pathlib import Path
import json

# Add src to path
REPO_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
SRC_DIR = REPO_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))
DATA_DIR = Path(REPO_ROOT,'data/wellness_centre')
OUTPUTS_DIR = Path(REPO_ROOT, "user-data/outputs")

print(f"Repository: {REPO_ROOT}")
print(f"Source: {SRC_DIR}")
print(f"Data: {DATA_DIR}")
print(f"Outputs: {OUTPUTS_DIR}")

Repository: /home/jovyan/work/ERP/emt
Source: /home/jovyan/work/ERP/emt/src
Data: /home/jovyan/work/ERP/emt/data/wellness_centre
Outputs: /home/jovyan/work/ERP/emt/user-data/outputs


In [2]:
# Import our migration toolkit
from orchestration import MigrationOrchestrator

# Initialize
data_dir = Path(DATA_DIR)
orchestrator = MigrationOrchestrator(data_dir)

print("✓ Migration orchestrator initialized")

✓ Migration orchestrator initialized


In [3]:
# Process ALL records (no limit)
print("Running FULL migration - processing all records...")
print("="*70)

results = orchestrator.process_batch()  # ← No limit = ALL records

# Generate report
print("\n")
report = orchestrator.generate_report(results)
print(report)

Running FULL migration - processing all records...
Starting batch processing (limit: ALL)...

1. Loading CSV data...
   Loaded 25 events
   Loaded 54 room bookings
   Loaded 103 egg sales

2. Generating invoices...
   Generated 25 event invoices
   Generated 54 room invoices
   Generated 103 egg invoices
   Total: 182 invoices

3. Calculating totals...
   Event revenue: KES 1,969,680.00
   Room revenue: KES 718,040.00
   Egg revenue: KES 136,840.00
   Total revenue: KES 2,824,560.00


WELLNESS CENTRE MIGRATION REPORT

DATA LOADED:
  Events:          25
  Room Bookings:   54
  Egg Sales:      103

INVOICES GENERATED:
  Event Invoices:   25
  Room Invoices:    54
  Egg Invoices:    103
  Total:           182

REVENUE BREAKDOWN:
  Events:
    Subtotal: KES 1,698,000.00
    Tax:      KES 271,680.00
    Total:    KES 1,969,680.00

  Rooms:
    Subtotal: KES 619,000.00
    Tax:      KES 99,040.00
    Total:    KES 718,040.00

  Eggs:
    Subtotal: KES 136,840.00
    Tax:      KES 0.00
    To

In [4]:
# Export to JSON
output_dir = Path(OUTPUTS_DIR)
export_info = orchestrator.export_erpnext_payloads(results, output_dir)

print(f"\n✓ Exported {export_info['count']} invoices")
print(f"✓ File location: {export_info['file']}")


✓ Exported 182 invoices
✓ File location: /home/jovyan/work/ERP/emt/user-data/outputs/sales_invoices.json


In [5]:
from frappeclient import FrappeClient
import os
import csv

# Connect to ERPNext
# Internal Docker URL — avoids Cloudflare round-trip latency.
# Fall back to external domain only if internal URL fails.
ERPNEXT_URL    = "http://erpnext-frontend:8080"
ERPNEXT_DOMAIN = "well.rosslyn.cloud"  # used as Host header for internal routing

# Path to API credentials file
# CSV format: two columns named 'api_key' and 'api_secret', one row of values
API_KEYS_FILE = "/home/jovyan/work/ERP/Wellness Centre/frappe_api_keys.csv"

# ─────────────────────────────────────────────────────────────
# Credential loading — CSV → environment variable → empty
# ─────────────────────────────────────────────────────────────
API_KEY    = ""
API_SECRET = ""

if os.path.exists(API_KEYS_FILE):
    with open(API_KEYS_FILE, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            API_KEY    = row.get("api_key", "").strip()
            API_SECRET = row.get("api_secret", "").strip()
            break  # only read the first data row
    print(f"✓ Credentials loaded from {API_KEYS_FILE}")
else:
    # Fall back to environment variables
    API_KEY    = os.environ.get("ERPNEXT_API_KEY", "")
    API_SECRET = os.environ.get("ERPNEXT_API_SECRET", "")
    if API_KEY:
        print("✓ Credentials loaded from environment variables")
    else:
        print(f"⚠  Credentials file not found: {API_KEYS_FILE}")
        print("   Set ERPNEXT_API_KEY / ERPNEXT_API_SECRET environment variables,")
        print("   or create the CSV file, or paste keys directly into API_KEY / API_SECRET above.")

# ─────────────────────────────────────────────────────────────
# Validation
# ─────────────────────────────────────────────────────────────
problems = []
if not API_KEY:
    problems.append("API_KEY is empty")
if not API_SECRET:
    problems.append("API_SECRET is empty")
if not os.path.exists(DATA_DIR):
    problems.append(f"DATA_DIR not found: {DATA_DIR}")

if problems:
    print("\n⚠  Configuration problems:")
    for p in problems:
        print(f"   • {p}")
else:
    print(f"\n✓ Configuration looks good")
    print(f"  URL:      {ERPNEXT_URL}")
    print(f"  Domain:   {ERPNEXT_DOMAIN}")
    print(f"  Data dir: {DATA_DIR}")
    print(f"  API key:  {'set (' + API_KEY[:6] + '...)' if API_KEY else 'MISSING'}")

# Connect using Docker network pattern
client = FrappeClient(ERPNEXT_URL)
client.authenticate(API_KEY, API_SECRET)

# CRITICAL: Add Host header for Docker network routing
client.session.headers.update({"Host": ERPNEXT_DOMAIN})

# Test connection
try:
    test = client.get_list("Customer", fields=["name"], limit_page_length=1)
    print(f"✓ Connected to ERPNext at {ERPNEXT_URL}")
    print(f"  Host header: {ERPNEXT_DOMAIN}")
    print(f"  Connection: Working")
except Exception as e:
    print(f"✗ Connection failed: {e}")
    print("\nFallback: Try external URL")
    print("  ERPNEXT_URL = 'https://well.rosslyn.cloud'")
    print("  Remove Host header line")

✓ Credentials loaded from /home/jovyan/work/ERP/Wellness Centre/frappe_api_keys.csv

✓ Configuration looks good
  URL:      http://erpnext-frontend:8080
  Domain:   well.rosslyn.cloud
  Data dir: /home/jovyan/work/ERP/emt/data/wellness_centre
  API key:  set (c7edf8...)
✓ Connected to ERPNext at http://erpnext-frontend:8080
  Host header: well.rosslyn.cloud
  Connection: Working


In [7]:
# Copy new files to toolkit first (in Jupyter terminal):
# cd ~/work/ERP/emt
# cp /mnt/user-data/outputs/erpnext_submitter.py src/orchestration/
# cp /mnt/user-data/outputs/erpnext_connection.py src/orchestration/

from orchestration.erpnext_submitter import ERPNextSubmitter
from orchestration.master_data_creator import MasterDataCreator

# Initialize
submitter = ERPNextSubmitter(client)
master_creator = MasterDataCreator(client, "Wellness Centre")

print("✓ ERPNext submitter initialized")
print("✓ Master data creator initialized")

✓ ERPNext submitter initialized
✓ Master data creator initialized


In [8]:
# Get all invoices from migration
all_invoices = (
    results['invoices']['event_invoices'] +
    results['invoices']['room_invoices'] +
    results['invoices']['egg_invoices']
)

print(f"Creating master data for {len(all_invoices)} invoices...")
print("="*70)

# Create customers, items, verify accounts
master_results = master_creator.create_all_masters(all_invoices)

# Check if ready
customers_ok = master_results['customers']['created'] > 0
items_ok = master_results['items']['created'] > 0

if customers_ok and items_ok:
    print("\n✓✓✓ All prerequisites met - ready to submit!")
else:
    print("\n⚠ Review master data results above")

Creating master data for 182 invoices...
CREATING MASTER DATA

Creating customers from invoices...
Found 32 unique customers
  ✓ Customer exists: Alice Kahiga
  ✓ Customer exists: Apiyo Kibe
  ✓ Customer exists: Atieno Ndirangu
  ✓ Created customer: Beatrice Mwende
  ✓ Created customer: Caroline Muthama
  ✓ Created customer: Charles Ogutu
  ✓ Customer exists: Damaris Sigei
  ✓ Created customer: David Mutua
  ✓ Created customer: Diana Cherop
  ✓ Customer exists: Dorothy Barasa
  ✓ Created customer: Esther Nyokabi
  ✓ Created customer: Faith Chebet
  ✓ Created customer: George Mwangi
  ✓ Created customer: Grace Akinyi
  ✓ Created customer: Henry Odhiambo
  ✓ Created customer: James Karanja
  ✓ Created customer: Janet Mukami
  ✓ Customer exists: Juma Korir
  ✓ Customer exists: Kiprono Muhoro
  ✓ Customer exists: Lagat Nyambura
  ✓ Created customer: Lucy Wanjiku
  ✓ Customer exists: Maina Mundia
  ✓ Created customer: Mercy Auma
  ✓ Customer exists: Murimi Onyango
  ✓ Customer exists: Mwang

In [9]:
# Submit all invoices
print(f"Submitting {len(all_invoices)} invoices to ERPNext...")
print(f"  Duplicate check: Enabled")
print(f"  Auto-submit: Disabled (keeping as Draft)")
print("="*70)

result = submitter.submit_invoices(
    invoices=all_invoices,
    check_duplicates=True,   # Skip if already exists
    auto_submit=False        # Keep as Draft for review
)

# Print detailed summary
print(result.summary())

INFO     | Submitting 182 invoices to ERPNext...
INFO     |   Check duplicates: True
INFO     |   Auto-submit: False


Submitting 182 invoices to ERPNext...
  Duplicate check: Enabled
  Auto-submit: Disabled (keeping as Draft)


INFO     |   Progress: 10/182 (✓ 0, ⊘ 0, ✗ 10)
INFO     |   Progress: 20/182 (✓ 0, ⊘ 0, ✗ 20)
INFO     |   Progress: 30/182 (✓ 0, ⊘ 0, ✗ 30)
INFO     |   Progress: 40/182 (✓ 0, ⊘ 0, ✗ 40)
INFO     |   Progress: 50/182 (✓ 0, ⊘ 0, ✗ 50)
INFO     |   Progress: 60/182 (✓ 0, ⊘ 0, ✗ 60)
INFO     |   Progress: 70/182 (✓ 0, ⊘ 0, ✗ 70)
INFO     |   Progress: 80/182 (✓ 0, ⊘ 0, ✗ 80)
INFO     |   Progress: 90/182 (✓ 0, ⊘ 0, ✗ 90)
INFO     |   Progress: 100/182 (✓ 0, ⊘ 0, ✗ 100)
INFO     |   Progress: 110/182 (✓ 0, ⊘ 0, ✗ 110)
INFO     |   Progress: 120/182 (✓ 0, ⊘ 0, ✗ 120)
INFO     |   Progress: 130/182 (✓ 0, ⊘ 0, ✗ 130)
INFO     |   Progress: 140/182 (✓ 0, ⊘ 0, ✗ 140)
INFO     |   Progress: 150/182 (✓ 0, ⊘ 0, ✗ 150)
INFO     |   Progress: 160/182 (✓ 0, ⊘ 0, ✗ 160)
INFO     |   Progress: 170/182 (✓ 0, ⊘ 0, ✗ 170)
INFO     |   Progress: 180/182 (✓ 0, ⊘ 0, ✗ 180)
INFO     |   Progress: 182/182 (✓ 0, ⊘ 0, ✗ 182)



IMPORT SUMMARY — Sales Invoice
  Total records:  182
  Succeeded:      0
  Skipped:        0  (already existed)
  Failed:         182
  Duration:       230s

FAILURES (182):
  - [Dorothy Barasa] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application
  - [Mwangi Naliaka] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application
  - [Owuor Muchiri] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application
  - [Alice Kahiga] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application
  - [Lagat Nyambura] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application
  - [Kiprono Muhoro] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application
  - [Maina Mundia] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", l

In [10]:
# Cell: Inspect first failure in detail
if result.failures:
    first_failure = result.failures[0]
    
    print("FIRST FAILURE - FULL ERROR:")
    print("="*70)
    print(f"Customer: {first_failure['record_id']}")
    print(f"\nFull Error Message:")
    print(first_failure['error'])  # Complete error, not truncated
    print("="*70)

FIRST FAILURE - FULL ERROR:
Customer: Dorothy Barasa

Full Error Message:
["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"app


In [11]:
# Cell: Inspect what we're trying to send
sample_invoice = all_invoices[0]

print("SAMPLE INVOICE PAYLOAD:")
print("="*70)
print(f"Customer: {sample_invoice.customer}")
print(f"Date: {sample_invoice.posting_date}")
print(f"Items: {len(sample_invoice.items)}")
print(f"Total: {sample_invoice.grand_total}")
print()

# See ERPNext format
import json
payload = sample_invoice.to_erpnext_format()
print("ERPNext Payload (formatted):")
print(json.dumps(payload, indent=2, default=str))
print("="*70)

SAMPLE INVOICE PAYLOAD:
Customer: Dorothy Barasa
Date: 2024-03-16
Items: 1
Total: KES 40,600.00

ERPNext Payload (formatted):
{
  "doctype": "Sales Invoice",
  "customer": "Dorothy Barasa",
  "customer_name": "Dorothy Barasa",
  "posting_date": "2024-03-16",
  "due_date": "2024-03-31",
  "items": [
    {
      "item_name": "Event Venue Hire - Dorothy's Nephew's 10th Birthday",
      "description": "Event Venue Hire - Dorothy's Nephew's 10th Birthday",
      "qty": 1.0,
      "rate": 35000.0,
      "amount": 35000.0,
      "uom": "Nos",
      "item_code": "Event Venue Hire"
    }
  ],
  "taxes": [
    {
      "charge_type": "On Net Total",
      "account_head": "VAT - WC",
      "description": "VAT @ 16%",
      "rate": 16.0,
      "tax_amount": 5600.0
    }
  ],
  "remarks": "Birthday Party event: Dorothy's Nephew's 10th Birthday (40 guests)"
}
